In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Fixed trajectory payload G

G: one prespecified duck prompt and the original seed 2026092501. Source preparation is exactly one PAYLOAD_MULTI 50-step native trajectory and one native Wan decode, saved with RGB SHA before comparison. This is a new prompt case, not a new independent seed sample. No reselection or generation retry. P0/P1/M1/M05; FULL181, CROP177 [1:178], SHORT89 [37:126]. SHORT89 searches offsets 0..92 and uses the adopted public length support R22 (660 votes/bit); other views use R44 (1320 votes/bit). Original coordinates, signs, bit mapping and Counter tie decisions are unchanged.

One complete MP4 save/read per condition, then slices without another codec. P1/P0 compatibility is shown first. Quality references P0 and P1 are explicit; condition-P1 residual differences are distinct from video self-difference ratios and are not perceptual flicker measures. Blind receivers receive only video, key and public protocol. All candidates, ties, missing and failed slots remain; truth/bit errors join only after sealing blind readouts. FULL has one geometry candidate; repeated payload success is not synchronization evidence. No threshold, scan or science PASS.

Fixed source prompt: locked camera, a yellow rubber duck slowly drifting left to right on calm water, steady soft daylight, no people, no cuts

Comparison denominator: {"source_cases": 1, "conditions": 4, "observations": 12, "keys": 2, "sync_readouts": 24, "sync_candidate_scores": 792, "sync_candidate_tubelet_rows": 19280, "payload_reads": 24, "sync_posthoc": 24, "payload_posthoc": 24, "quality": 24, "rasters": 4, "mp4_save": 4, "mp4_read": 4}

Separate source preparation budget: {"generation_load": 1, "trajectory": 1, "transformer_conditional": 50, "transformer_unconditional": 50, "native_step": 50, "local_control": 25, "payload_gradient": 25, "pilot_gradient": 0, "wan_vae_load": 1, "wan_decode": 1, "raster_save": 1}

Published source: 9daf21321a34a710345cb0251241f148b065f546

In [ ]:
SOURCE_SHA = '9daf21321a34a710345cb0251241f148b065f546'
FIXED = {'source_cases': 1, 'conditions': 4, 'observations': 12, 'keys': 2, 'sync_readouts': 24, 'sync_candidate_scores': 792, 'sync_candidate_tubelet_rows': 19280, 'payload_reads': 24, 'sync_posthoc': 24, 'payload_posthoc': 24, 'quality': 24, 'rasters': 4, 'mp4_save': 4, 'mp4_read': 4}
from pathlib import Path
import datetime,hashlib,json,os,signal,subprocess,sys,time,traceback
if SOURCE_SHA is None:
    raise RuntimeError('UNPUBLISHED_DRAFT: publish reviewed source and rebuild with its immutable SHA before Run all')
if not isinstance(SOURCE_SHA,str) or len(SOURCE_SHA)!=40:
    raise RuntimeError('immutable 40-character source SHA required')
STAMP=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=Path('/content/drive/MyDrive/Video-WM/Trajectory-Payload-G-V1')/STAMP
OUTPUT.mkdir(parents=True,exist_ok=False)
RUN_OUTPUT=OUTPUT/'fixed_reference'
REPO=Path('/content/SC-SSTW-TRAJECTORY-PAYLOAD-G-V1-'+STAMP)
PYTHON=sys.executable
def write_json(path,value):
    tmp=path.with_suffix(path.suffix+'.tmp');tmp.write_text(json.dumps(value,indent=2)+'\n');os.replace(tmp,path)
def failed(stage,exc):
    captured_traceback=''.join(traceback.format_exception(type(exc),exc,exc.__traceback__))
    write_json(OUTPUT/'setup_failure.json',dict(stage=stage,error=f'{type(exc).__name__}: {exc}',traceback=captured_traceback,fixed_denominator=FIXED))
def logged(command,stage,cwd=None,check=True):
    child=None;code=None;primary=None;primary_tb=None;cleanup_errors=[]
    def retain_cleanup_error(label,exc):
        nonlocal primary,primary_tb
        if primary is None and not isinstance(exc,Exception):
            primary=exc;primary_tb=exc.__traceback__
        else:
            cleanup_errors.append(label+': '+repr(exc))
    try:
        with (OUTPUT/'execution.log').open('a') as log:
            log.write('COMMAND '+repr(command)+'\n');log.flush()
            popen_group={'start_new_session':True} if os.name=='posix' else {}
            child=subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,**popen_group)
            try:
                for line in child.stdout:
                    print(line,end='',flush=True);log.write(line);log.flush()
                code=child.wait()
            except BaseException as exc:
                primary=exc;primary_tb=exc.__traceback__
            finally:
                if child is not None:
                    if os.name=='posix':
                        pgid=child.pid
                        try:os.killpg(pgid,signal.SIGTERM)
                        except ProcessLookupError:pass
                        except BaseException as exc:retain_cleanup_error('group SIGTERM',exc)
                        try:
                            deadline=time.monotonic()+5.0
                            while time.monotonic()<deadline:
                                try:os.killpg(pgid,0)
                                except ProcessLookupError:break
                                except BaseException as exc:
                                    retain_cleanup_error('group probe',exc);break
                                time.sleep(0.05)
                        except BaseException as exc:
                            retain_cleanup_error('group TERM grace',exc)
                        finally:
                            try:os.killpg(pgid,signal.SIGKILL)
                            except ProcessLookupError:pass
                            except BaseException as exc:retain_cleanup_error('group SIGKILL',exc)
                    elif child.poll() is None:
                        try:child.terminate()
                        except ProcessLookupError:pass
                        except BaseException as exc:retain_cleanup_error('terminate',exc)
                    try:
                        try:code=child.wait(timeout=10)
                        except subprocess.TimeoutExpired:
                            try:child.kill()
                            except ProcessLookupError:pass
                            except BaseException as exc:retain_cleanup_error('kill',exc)
                            try:code=child.wait(timeout=10)
                            except BaseException as exc:retain_cleanup_error('wait after kill',exc)
                        except BaseException as exc:retain_cleanup_error('wait',exc)
                    finally:
                        if child.stdout is not None:
                            try:child.stdout.close()
                            except BaseException as exc:retain_cleanup_error('stdout close',exc)
            if primary is None and not cleanup_errors:
                log.write('EXIT '+str(code)+'\n');log.flush()
    except BaseException as wrapper_error:
        if primary is None:
            primary=wrapper_error;primary_tb=wrapper_error.__traceback__
        elif wrapper_error is not primary:
            cleanup_errors.append('log wrapper/close: '+repr(wrapper_error))
    if primary is None and cleanup_errors:
        primary=RuntimeError('process cleanup failed: '+'; '.join(cleanup_errors))
        primary_tb=primary.__traceback__
    if primary is None and check and code:
        primary=subprocess.CalledProcessError(code,command)
        primary_tb=primary.__traceback__
    if primary is not None:
        if cleanup_errors and hasattr(primary,'add_note'):
            primary.add_note('secondary cleanup errors: '+'; '.join(cleanup_errors))
        try:failed(stage,primary)
        except BaseException as record_error:
            if hasattr(primary,'add_note'):primary.add_note('failure record error: '+repr(record_error))
        raise primary.with_traceback(primary_tb)
    return code
write_json(OUTPUT/'setup_receipt.json',dict(status='SETUP_STARTED',source_sha=SOURCE_SHA,fixed_denominator=FIXED))


In [ ]:
try:
    logged(['git','clone','--filter=blob:none','https://github.com/RICHAAARC/SC-SSTW.git',str(REPO)],'SOURCE_CLONE')
    logged(['git','-C',str(REPO),'fetch','origin',SOURCE_SHA],'SOURCE_FETCH')
    logged(['git','-C',str(REPO),'checkout','--detach',SOURCE_SHA],'SOURCE_CHECKOUT')
    actual=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
    dirty=subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True)
    if actual!=SOURCE_SHA or dirty:raise RuntimeError('source SHA/clean checkout mismatch')
    import shutil
    if any(shutil.which(name) is None for name in ('ffmpeg','ffprobe')):
        logged(['apt-get','update'],'MEDIA_TOOL_UPDATE',check=False)
        logged(['apt-get','install','-y','ffmpeg'],'MEDIA_TOOL_REPAIR',check=False)
    if logged([PYTHON,'-c','import torch; print(torch.__version__)'],'TORCH_PROBE',check=False):
        logged([PYTHON,'-m','pip','install','--upgrade','torch'],'TORCH_REPAIR')
    probe='from diffusers import AutoencoderKL,AutoencoderKLWan,WanPipeline; import sentencepiece,ftfy; print(WanPipeline.__name__)'
    if logged([PYTHON,'-c',probe],'VAE_IMPORT_PROBE',check=False):
        logged([PYTHON,'-m','pip','install','diffusers>=0.35.1,<0.40','transformers>=4.45,<5','accelerate>=0.34','safetensors>=0.4.5','sentencepiece','ftfy'],'VAE_DEPENDENCY_REPAIR')
    dependency_code=logged([PYTHON,'-m','pip','check'],'DEPENDENCY_REPORT',check=False)
    (OUTPUT/'environment_freeze.txt').write_text(subprocess.check_output([PYTHON,'-m','pip','freeze'],text=True))
    write_json(OUTPUT/'source_receipt.json',dict(source_sha=actual,clean=True))
    write_json(OUTPUT/'setup_receipt.json',dict(status='SETUP_COMPLETE',source_sha=actual,fixed_denominator=FIXED,dependency_check_returncode=dependency_code))
except Exception as exc:
    try:failed('SOURCE_OR_ENVIRONMENT',exc)
    except BaseException as record_error:
        if hasattr(exc,'add_note'):exc.add_note('failure record error: '+repr(record_error))
    raise


In [ ]:
try:
    from google.colab import userdata
    token=userdata.get('HF_TOKEN')
    if token:os.environ['HF_TOKEN']=token
except Exception:pass
finally:token=None
command=[PYTHON,'-u','-m','experiments.wan_state_clock.video_trajectory_payload_g_v1_run','--output',str(RUN_OUTPUT)]
try:
    returncode=logged(command,'FIXED_RUN',cwd=REPO,check=False)
    RESULT_PATH=RUN_OUTPUT/'result.json'
    write_json(OUTPUT/'execution_receipt.json',dict(command=command,returncode=returncode,result_path=str(RESULT_PATH),result_exists=RESULT_PATH.is_file()))
    if not RESULT_PATH.is_file():raise RuntimeError('No runner result; fixed denominator retained')
except Exception as exc:
    try:failed('RUNNER',exc)
    except BaseException as record_error:
        if hasattr(exc,'add_note'):exc.add_note('failure record error: '+repr(record_error))
    raise


In [ ]:
import statistics
result=json.loads(RESULT_PATH.read_text())
if result['source_sha']!=SOURCE_SHA or result['fixed_denominator']!=FIXED:raise RuntimeError('result identity mismatch')
print('Status:',result['status'],'Counts:',result['counts'])
print('Comparison calls:',result['calls'])
print('Source preparation:',{k:v for k,v in result['source_preparation'].items() if k not in ('steps','trajectory_receipt')})
print('Source steps complete:',sum(row['status']=='COMPLETE' for row in result['source_preparation']['steps']))
print('Source preparation calls:',result['source_calls'])
print('Frozen source identity:',result['source_protocol'])
for condition,row in result['writers'].items():
    print('Writer:',condition,'status=',row['status'],'path=',row.get('path'),
          'constraint/constructed/applied L2=',row.get('constraint_increment_l2',row.get('raw_target_delta_l2')),
          row.get('constructed_delta_l2'),row.get('applied_float32_delta_l2'),
          'full applied residual=',row.get('max_applied_full_residual'),'error=',row.get('error'))
def joined(group):
    for sid,row in sorted(result[group].items()):
        oid,key_id=sid.rsplit('/',1);obs=result['observations'][oid]
        value=dict(row);value.setdefault('condition',obs['condition_truth_join_only'])
        value.setdefault('view',obs['view_truth_join_only'])
        value.setdefault('key_role','REGISTERED' if key_id=='K0' else 'WRONG_KEY')
        yield sid,value
print('=== P1/P0 payload compatibility first ===')
for sid,row in joined('payload_posthoc'):
    if row['condition'] in ('P0_ORIGINAL_RGB','P1_FRAMEWISE_RECON'):
        print(sid,row['status'],row['key_role'],'bit_errors=',row.get('bit_errors'),'error=',row.get('error'))
print('=== All quality pairs: references are explicit ===')
for qid,row in result['quality'].items():
    print(qid,row)
print('=== All keys and views: truth joined after blind seal ===')
for sid,row in joined('sync_posthoc'):
    print('FULL geometry only' if row.get('full_singleton_geometry_only',row['view']=='FULL181') else 'Localization',
          sid,row['status'],row['condition'],row['view'],row['key_role'],
          'truth_score=',row.get('true_score'),'best_other=',row.get('best_other_score'),
          'gap=',row.get('true_score_gap'),'rank=',row.get('true_rank'),
          'unique_truth=',row.get('unique_truth'),'error=',row.get('error'),
          'all-offset readout=',result['sync_reads'][sid]['path'])
print('=== All payload reads; per-bit raw votes/margins/errors retained in result ===')
for sid,row in joined('payload_posthoc'):
    bits=row.get('bit_rows',[])
    margins=[v['margin'] for v in bits];normalized=[v['normalized_margin'] for v in bits]
    print(sid,row['status'],row['condition'],row['view'],row['key_role'],
          'R=',result['payload_reads'][sid].get('R'),'bit_errors=',row.get('bit_errors'),
          'margin_min/median=',(min(margins),statistics.median(margins)) if margins else None,
          'normalized_min/median=',(min(normalized),statistics.median(normalized)) if normalized else None,
          'error=',row.get('error'))
from IPython.display import Video,display
for condition,row in result['transport'].items():
    video=row.get('events',{}).get('mp4',{});path=video.get('path')
    if video.get('status')=='SAVED' and isinstance(path,str) and Path(path).is_file():
        try:print('MP4:',condition);display(Video(str(path),embed=True))
        except Exception as preview_error:print('Preview unavailable:',preview_error)
print('Failures:',result['failures']);print('Evidence ceiling:',result['evidence_ceiling']);print('Result:',RESULT_PATH)
